# Simple RAG Application with LangGraph

A simple sequential RAG system using LangGraph for answering HR policy questions. This notebook uses a single vector database built from company leave and promotion/bonus policy documents. There is no user database, no agent tooling, and no memory. The pipeline is a straightforward retrieve-then-generate flow.

## 1. Install Required Libraries

In [1]:
!pip install -q langgraph langchain langchain-openai langchain-community
!pip install -q chromadb
!pip install -q langchain-chroma
!pip install -q pypdf

## 2. Import Libraries

In [2]:
import os
from typing import TypedDict
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

import warnings
warnings.filterwarnings("ignore")

## 3. Configuration

In [3]:
# Set OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI model
llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 4. Setup Vector Database

Both company policy documents (leave policies and promotion/bonus policies) are loaded, chunked, and indexed into a single vector database.

### 4.1 Initialize Text Splitter

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

print("Text splitter initialized!")

Text splitter initialized!


### 4.2 Load and Merge Both Policy Documents

In [ ]:
LEAVE_POLICY_PDF_PATH = "/home/mani/PatronusAI/agentic_memory/Company_Leave_Policies_Extended.pdf"
PROMOTION_BONUS_PDF_PATH = "/home/mani/PatronusAI/agentic_memory/Company_Promotion_Bonus_Policies_Enterprise_Grade.pdf"

def load_merged_vectorstore(pdf_paths):
    all_documents = []
    for pdf_path in pdf_paths:
        if not os.path.exists(pdf_path):
            print(f"Warning: PDF not found at {pdf_path}")
            continue
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        all_documents.extend(documents)
        print(f"Loaded {len(documents)} pages from {os.path.basename(pdf_path)}")
    
    if not all_documents:
        print("No documents loaded!")
        return None
    
    split_docs = text_splitter.split_documents(all_documents)
    
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        collection_name="company_policies",
        persist_directory="./chroma_db_rag"
    )
    
    print(f"\nTotal: {len(all_documents)} pages, split into {len(split_docs)} chunks")
    return vectorstore

vectorstore = load_merged_vectorstore([LEAVE_POLICY_PDF_PATH, PROMOTION_BONUS_PDF_PATH])
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) if vectorstore else None

Loaded 7 pages from Company_Leave_Policies_Extended.pdf
Loaded 9 pages from Company_Promotion_Bonus_Policies_Enterprise_Grade.pdf

Total: 16 pages, split into 30 chunks


## 5. Define Graph State

In [ ]:
class State(TypedDict):
    question: str
    context: str
    answer: str

## 6. Define Graph Nodes

### 6.1 Retrieve Node

In [ ]:
def retrieve(state: State) -> State:
    """Retrieve relevant documents from vector database."""
    question = state["question"]
    
    if retriever is None:
        return {**state, "context": "No documents loaded."}
    
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    return {**state, "context": context}

### 6.2 Generate Node

In [ ]:
def generate(state: State) -> State:
    """Generate answer based on retrieved context."""
    question = state["question"]
    context = state["context"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful HR assistant. Answer the question based on the provided policy context. If the context does not contain enough information to answer, say so."),
        ("human", """Context:
{context}

Question: {question}

Provide a clear and concise answer based on the context above.""")
    ])
    
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    
    return {**state, "answer": response.content}

## 7. Build Sequential RAG Graph

In [ ]:
workflow = StateGraph(State)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

print("Sequential RAG graph compiled!")

## 8. Query Function

In [ ]:
def query_rag(question: str) -> str:
    """Query the RAG system.
    
    Args:
        question: Question about company policies
    """
    result = app.invoke({"question": question})
    return result["answer"]

## 9. Example Usage

In [ ]:
# General leave policy question
answer = query_rag("How many days of annual leave are employees entitled to?")
print("Q: How many days of annual leave are employees entitled to?")
print(f"A: {answer}")
print("\n" + "="*80 + "\n")

In [ ]:
# Bonus policy question
answer = query_rag("What are the target bonus levels for each employee tier?")
print("Q: What are the target bonus levels for each employee tier?")
print(f"A: {answer}")
print("\n" + "="*80 + "\n")

In [ ]:
# Maternity leave question
answer = query_rag("What is the maternity leave policy?")
print("Q: What is the maternity leave policy?")
print(f"A: {answer}")
print("\n" + "="*80 + "\n")

## 10. Limitations: Problems Without Memory

The simple RAG pipeline above works well for standalone factual questions. But it has no notion of who is asking, no conversation history, and no ability to learn or adapt. The following examples demonstrate where this breaks down.

### 10.1 Cannot Personalize: The System Does Not Know Who You Are

Without a user database or memory, the system cannot tailor answers to a specific employee. It can only give generic policy information.

In [ ]:
answer = query_rag("How many days of annual leave do I get?")
print("Q: How many days of annual leave do I get?")
print(f"A: {answer}")
print("\n--- The system lists all tiers because it has no idea who 'I' am. ---")
print("--- A memory-enabled agent would know the user's seniority level ---")
print("--- and give a direct, personalized answer. ---")

### 10.2 Cannot Handle Follow-Ups: Every Query Starts From Zero

Each call to `query_rag` is completely independent. The system has no conversation history, so follow-up questions that reference earlier turns fail.

In [ ]:
# Turn 1
answer1 = query_rag("What is the maternity leave policy?")
print("Turn 1 Q: What is the maternity leave policy?")
print(f"Turn 1 A: {answer1}")
print("\n" + "-"*80 + "\n")

# Turn 2: Follow-up referencing Turn 1
answer2 = query_rag("Can I combine it with annual leave?")
print("Turn 2 Q: Can I combine it with annual leave?")
print(f"Turn 2 A: {answer2}")
print("\n--- The system does not know what 'it' refers to. ---")
print("--- It lost all context from the previous question. ---")

### 10.3 Cannot Remember User Preferences Across Sessions

If a user shares information about themselves, the system immediately forgets it. There is no mechanism to store or recall anything between queries.

In [ ]:
# Session 1: User shares their role
answer1 = query_rag("I am a Senior Software Engineer in the Engineering department. What leave policies apply to me?")
print("Session 1 Q: I am a Senior Software Engineer... What leave policies apply to me?")
print(f"Session 1 A: {answer1}")
print("\n" + "-"*80 + "\n")

# Session 2: Ask a follow-up expecting the system to remember the role
answer2 = query_rag("Am I eligible for a bonus?")
print("Session 2 Q: Am I eligible for a bonus?")
print(f"Session 2 A: {answer2}")
print("\n--- The system forgot that the user is a Senior Software Engineer. ---")
print("--- It cannot connect the two sessions. ---")

### 10.4 Cannot Handle Corrections or Updates

Without memory, the system has no way to track changes. If a user tells the system their role has changed, it cannot store or apply that correction.

In [ ]:
# User shares updated information
answer1 = query_rag("I was recently promoted from Mid-level to Senior. What bonus am I now eligible for?")
print("Q: I was recently promoted from Mid-level to Senior. What bonus am I now eligible for?")
print(f"A: {answer1}")
print("\n" + "-"*80 + "\n")

# Next query: system has already forgotten the promotion
answer2 = query_rag("What is my bonus eligibility?")
print("Q: What is my bonus eligibility?")
print(f"A: {answer2}")
print("\n--- The system already forgot the promotion from the previous query. ---")
print("--- A memory-enabled agent would store the update and use it going forward. ---")